# Train Word2vec and Explore Word Relationships - Exercise

**Business question:** how can we learn useful word relationships from emails before building a spam detector?

We will train word2vec, choose words we want to explore, and draw a map of their learned relationships. Page 17 of *Slides_MSC_Data_Science_Week_08_EN.pdf* shows one example; your map can use different words.

## What you will learn

- Explain how word2vec learns from words that appear near each other.
- Train a model that represents each word as a list of numbers, called a **word vector**.
- Draw a word map and check whether it agrees with cosine similarity.
- Explain what the picture can and cannot tell us about spam detection.

## The email data

We use **`email_spam_text.csv.gz`**, which contains 30,000 emails from the original experiment. Pandas opens this compressed file directly. No extra download or upload is needed in Codespaces.

The emails are already divided into three groups:

- **Training: 18,000 emails.** Word2vec learns from these words.
- **Validation: 6,000 emails.** Reserved for later model choices.
- **Test: 6,000 emails.** Reserved for the final check of a spam detector.

**Word2vec does not use the spam labels.** It learns from neighbouring words, not from whether an email is spam. This activity trains word vectors; it does not yet train a spam detector.

The file `email_spam.csv` is different: it contains counts and other numerical features, not the original words. We cannot use it to train word2vec.

All word vectors come from the model we train in this notebook. We do not load a separate file of saved word vectors.

The supplied email text is approved for the course's intended audience. Do not display private messages or share the data beyond its permitted use. See the [data notes](../../../00_Data/nlp_data_readme.md) for details.

**Your tasks:** complete the three `None` values marked `# TODO`, then answer the questions. Run the cells from top to bottom. Start with the [concept notebook](text_embeddings_concept.ipynb) if you need a reminder about word vectors.

## 1. Load the libraries and choose settings

We train our own model using PyTorch on the CPU. We do not download a pretrained model.

Each word will have **100 numbers**. Training makes **five passes through the training emails**. One pass is called an **epoch**.

The **random seed** makes random choices repeatable when the software and data stay the same. We use seed 7, as in the original experiment. Different software versions can still give different results.

Keep `MAX_MESSAGES = 30_000` so the supplied data groups stay unchanged. You can try fewer epochs to explore the code, but this changes the experiment. A live chart will show progress during training.

In [ ]:
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

DATA_ROOT = Path('../../../00_Data').resolve()
RAW_EMAIL_PATH = Path(os.environ.get('SPAM_DATA_PATH', str(DATA_ROOT / 'email_spam_text.csv.gz'))).expanduser()
SEED = 7
MAX_MESSAGES = 30_000
VECTOR_SIZE = 100
EPOCHS = 5
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
display(pd.Series({name: version(name) for name in ['numpy', 'torch', 'scikit-learn']},
                  name='Installed version'))

## 2. Load the emails

We reuse the data-reading code from the Neural Networks material. It keeps the supplied training, validation and test groups in their original order.

For training, we split email subjects and bodies into small pieces called **tokens**. Here these are lowercase words, contractions such as `don't`, or sequences of digits.

We show counts, not real messages. We also keep emails separate: a word at the end of one email is never paired with a word in the next.

In [ ]:
if not RAW_EMAIL_PATH.is_file():
    raise FileNotFoundError(
        'Set RAW_EMAIL_PATH or SPAM_DATA_PATH to the authorised raw message/label CSV. '
        'The supplied default is 00_Data/email_spam_text.csv.gz.')
if not {'message', 'label'}.issubset(pd.read_csv(RAW_EMAIL_PATH, nrows=0).columns):
    raise ValueError('Word2vec needs raw message and label columns, not the feature-only email CSV.')
sys.path.insert(0, str(DATA_ROOT.parent / '08_Neural_Networks/HS2026'))
from enron_data import load_email_splits

train, validation, test = load_email_splits(RAW_EMAIL_PATH, max_messages=MAX_MESSAGES, seed=SEED)
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?|\d+")
training_text = train.subject.fillna('') + '\n' + train.body.fillna('')
training_tokens = training_text.map(lambda text: TOKEN_PATTERN.findall(text.casefold())).tolist()
display(pd.Series({'train emails': len(train), 'validation emails': len(validation),
                   'test emails': len(test), 'training tokens': sum(map(len, training_tokens))},
                  name='Data summary'))

## 3. Find words that appear near each other

Consider the made-up phrase **"claim your prize"**. It gives us word pairs such as `claim / prize`. Across many emails, the model learns from repeated patterns like these.

The word we start from is the **centre word**. A nearby word is a **context word**.

The model's list of known words is its **vocabulary**. We keep words that appear at least five times, with a maximum of 30,000 different words. Very common words are sometimes skipped so that they do not dominate training.

The supplied code prepares pairs from short blocks of text, looking up to five positions on either side. You do not need to change this code. The vocabulary count is enough for our inspection; we do not print the full word list.

In [ ]:
word_counts = Counter(word for document in training_tokens for word in document)
vocabulary = sorted(
    (word for word, count in word_counts.items() if count >= 5),
    key=lambda word: (-word_counts[word], word),
)[:30_000]
if len(vocabulary) < 2:
    raise ValueError('Training text needs at least two words with frequency >= 5.')
word_to_index = {word: index for index, word in enumerate(vocabulary)}
counts = np.array([word_counts[word] for word in vocabulary], dtype=np.float64)
frequencies = counts / counts.sum()
keep_probability = np.minimum(1., np.sqrt(1e-3 / frequencies) + 1e-3 / frequencies)
noise_distribution = torch.tensor(counts ** 0.75 / (counts ** 0.75).sum(), dtype=torch.float32)
training_ids = [
    np.array([word_to_index[word] for word in document if word in word_to_index], dtype=np.int64)
    for document in training_tokens
]
print('Training vocabulary size:', len(vocabulary))

In [ ]:
def pair_batches(documents, keep_probability, rng, batch_size=4096):
    pending, pending_count = [], 0
    for ids in documents:
        for start in range(0, len(ids), 100):
            span = ids[start:start + 100]
            span = span[rng.random(len(span)) < keep_probability[span]]
            pairs = []
            for distance in range(1, min(5, len(span) - 1) + 1):
                left, right = span[:-distance], span[distance:]
                pairs.extend([np.column_stack((left, right)), np.column_stack((right, left))])
            if not pairs:
                continue
            pending.append(np.concatenate(pairs))
            pending_count += len(pending[-1])
            if pending_count >= batch_size:
                combined = np.concatenate(pending)
                stop = len(combined) // batch_size * batch_size
                for offset in range(0, stop, batch_size):
                    yield combined[offset:offset + batch_size]
                pending = [combined[stop:]]
                pending_count = len(combined) - stop
    if pending_count:
        yield np.concatenate(pending)

## 4. Give the model a learning task

**Skip-gram** is a way to train word2vec using a centre word and nearby words. The model keeps two tables of numbers:

1. A table for the word we start from.
2. A table for the nearby word we compare it with.

A **dot product** combines the two vectors into a score. Training adjusts the numbers so that real neighbouring pairs tend to score higher than randomly paired words.

For each real pair, we draw five random comparison words. This is called **negative sampling**. It is faster than comparing every word with the entire vocabulary.

The sigmoid function turns each score into a value between zero and one. The **loss** measures how poorly the model separates real pairs from random pairs; lower is better for this task. These scores are **not spam probabilities**.

The slides use softmax to explain predictions across all possible neighbouring words. Our faster training method does not calculate that full set of probabilities.

The two `nn.Embedding` lines below create the tables. We will use the first table to draw the word map.

### The loss we actually calculate

`SkipGramNegativeSampling` contains **both the embedding tables and the loss calculation**. Calling `word2vec(centres, contexts, negatives)` runs its `forward()` method and returns the loss for that batch. There is no separate `.fit()` call: our training loop calculates the loss, calls `loss.backward()` to calculate gradients, and calls `optimizer.step()` to update the tables.

For one observed word pair and five randomly sampled comparison words, the loss is:

$$\ell = -\log\sigma(\mathbf{v}_w^\top\mathbf{u}_c) - \sum_{j=1}^{5}\log\sigma(-\mathbf{v}_w^\top\mathbf{u}_{n_j})$$

- $w$ is the centre word; $c$ is an observed nearby word; $n_j$ is comparison word number $j$.
- $\mathbf{v}_w$ comes from the centre-word table. $\mathbf{u}_c$ and $\mathbf{u}_{n_j}$ come from the context-word table.
- $\mathbf{v}^\top\mathbf{u}$ is the dot-product score, and $\sigma(z)=1/(1+e^{-z})$ is the sigmoid function.
- The first term encourages a **high score for the observed pair**. The second encourages **low scores for the sampled comparison pairs**. Randomly sampled words are not guaranteed to be unrelated.

For a batch of $B$ observed pairs, we take the average: $L_{\mathrm{batch}} = \frac{1}{B}\sum_{i=1}^{B}\ell_i$. This matches the return statement below: `positive` holds the observed-pair scores; `negative` holds the comparison-pair scores; `.sum(dim=1)` adds the five comparison losses; `.mean()` averages over the batch. `F.logsigmoid` calculates log-sigmoid in a numerically stable way.

This is binary cross-entropy with label **1 for observed pairs** and **0 for sampled comparison pairs**, summed across the six pairs for each centre/context example, then averaged over the batch. We learn the vectors that produce these scores. **Spam labels are not used.**

### How many parameters do we learn?

A parameter is one adjustable number. With vocabulary size $V$ and vector length $d$, the two tables contain **$2Vd$ trainable parameters**. In the supplied full run, $V=30{,}000$ and $d=100$:

| Table | Calculation | Parameters |
|---|---:|---:|
| Centre words | 30,000 x 100 | 3,000,000 |
| Context words | 30,000 x 100 | 3,000,000 |
| **Total** | **2 x 30,000 x 100** | **6,000,000** |

There are no bias terms or extra trainable layers. Negative sampling adds comparison examples, not parameters. The word map uses only the centre-word table. Choosing fewer words for the plot does not make the trained model smaller. The code after model creation reports the actual count for your run.

In [ ]:
class SkipGramNegativeSampling(nn.Module):
    def __init__(self, vocabulary_size: int, dimensions: int):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocabulary_size, dimensions, sparse=True)
        self.context_embeddings = nn.Embedding(vocabulary_size, dimensions, sparse=True)
        nn.init.uniform_(self.input_embeddings.weight, -0.5 / dimensions, 0.5 / dimensions)
        nn.init.zeros_(self.context_embeddings.weight)

    def forward(self, centres, contexts, negatives):
        vectors = self.input_embeddings(centres)
        positive = (vectors * self.context_embeddings(contexts)).sum(dim=1)
        negative = torch.bmm(
            self.context_embeddings(negatives), vectors.unsqueeze(2)
        ).squeeze(2)
        return -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()

### Check the starting point

Before learning begins, we give the model a fixed set of word pairs and record its loss. After training, we check the **same pairs** again. This lets us compare the model before and after learning.

These are training examples, so improvement here does not tell us how well the model handles new emails. It also does not tell us whether spam detection improved. For that, we would need a separate classifier comparison on emails not used for training.

In [ ]:
word2vec = SkipGramNegativeSampling(len(vocabulary), VECTOR_SIZE)
optimizer = torch.optim.SparseAdam(word2vec.parameters(), lr=0.01)
reference_pairs = next(pair_batches(
    training_ids, keep_probability, np.random.default_rng(SEED + 100_000)), None)
if reference_pairs is None:
    raise ValueError('No training word/context pairs survived subsampling.')
reference_pairs = torch.from_numpy(reference_pairs)
reference_negatives = torch.multinomial(
    noise_distribution, len(reference_pairs) * 5, replacement=True,
    generator=torch.Generator().manual_seed(SEED + 100_000),
).reshape(-1, 5)
with torch.no_grad():
    initial_pair_loss = word2vec(
        reference_pairs[:, 0], reference_pairs[:, 1], reference_negatives).item()
print(f'Untrained loss on fixed training pairs: {initial_pair_loss:.3f}')

In [ ]:
parameter_count = sum(parameter.numel() for parameter in word2vec.parameters()
                      if parameter.requires_grad)
print(f'Vocabulary: {len(vocabulary):,} words; vector length: {VECTOR_SIZE}')
print(f'Trainable parameters: {parameter_count:,}')

## 5. Let the model learn

A **batch** is a group of word pairs processed together. For each batch, the code:

1. Scores real pairs and random comparison pairs.
2. Calculates the loss.
3. Uses backpropagation to work out how to change the numbers.
4. Applies those changes with the optimizer.

`loss.backward()` works out the changes; `optimizer.step()` applies them.

Watch the live chart. It updates every 500 batches and at the end of each epoch. Each point shows the average loss since the previous update, giving equal weight to each word pair. The title shows the current epoch, the total pairs processed and the recent loss.

A downward trend suggests the model is getting better at this word-pair task. The line can move up and down because each batch contains different pairs. **It is not a chart of spam accuracy**, and we do not use test emails to choose settings.

### Why can the chart have repeating dips?

Each point measures a different section of the training data, not a fixed test. A lower value can mean that the current examples are easier, not just that the model improved. The emails stay in the **same order each epoch**; random word skipping and comparison-word sampling change, but we do not shuffle the emails. Easier sections can therefore produce dips at a similar position in every pass.

In the full example run, one epoch contains about **37.8 million word pairs**. Dips roughly that far apart suggest a repeated data-order effect. Repeated phrases or template-like emails could explain easier sections, but we would need to inspect those batches to confirm the cause. A rise after a dip does not necessarily mean the model forgot: the next examples may simply be harder.

Use the **whole-epoch average** to judge the broader trend. The supplied solution's averages fall from about 2.398 to 2.157, 2.106, 2.082 and 2.069. Our before/after comparison also uses the **same fixed training pairs and comparison words**, so their difficulty stays constant. Neither check establishes performance on unseen emails. An optional extension is to repeat that fixed-pair check after each epoch.

**Coding task:** replace `None` with `optimizer.step()` so the model's numbers are updated after each batch.

In [ ]:
progress_fig, progress_ax = plt.subplots(figsize=(8, 4))
progress_line, = progress_ax.plot([], [], marker='o', color='#00776e')
progress_ax.set_xlabel('Word/context pairs processed (millions)')
progress_ax.set_ylabel('Mean word-pair loss (recent batches)')
progress_ax.set_title('Word2vec learning progress: waiting for the first update')
progress_ax.grid(alpha=0.2)
progress_fig.tight_layout()
plt.close(progress_fig)
progress_display = display(progress_fig, display_id=True)
learning_progress = []

def show_learning_progress(epoch, processed_pairs, mean_loss):
    learning_progress.append({'processed pairs': processed_pairs, 'recent loss': mean_loss})
    progress_line.set_data(
        [row['processed pairs'] / 1_000_000 for row in learning_progress],
        [row['recent loss'] for row in learning_progress])
    progress_ax.relim()
    progress_ax.autoscale_view()
    progress_ax.set_ylim(bottom=0)
    progress_ax.set_title(f'Epoch {epoch + 1}/{EPOCHS} | {processed_pairs:,} pairs | recent loss {mean_loss:.3f}')
    progress_display.update(progress_fig)

In [ ]:
training_history = []
processed_pairs = 0
for epoch in range(EPOCHS):
    rng = np.random.default_rng(SEED + epoch)
    negative_rng = torch.Generator().manual_seed(SEED + epoch)
    total_loss, pair_count = 0., 0
    recent_loss, recent_pairs = 0., 0
    batch_count = 0
    for batch in pair_batches(training_ids, keep_probability, rng):
        pairs = torch.from_numpy(batch)
        negatives = torch.multinomial(
            noise_distribution, len(batch) * 5,
            replacement=True, generator=negative_rng,
        ).reshape(-1, 5)
        optimizer.zero_grad(set_to_none=True)
        loss = word2vec(pairs[:, 0], pairs[:, 1], negatives)
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite word2vec loss; stop and inspect training.')
        loss.backward()
        None  # TODO: update the embedding tables
        batch_loss = loss.item() * len(batch)
        total_loss += batch_loss
        pair_count += len(batch)
        recent_loss += batch_loss
        recent_pairs += len(batch)
        processed_pairs += len(batch)
        batch_count += 1
        if batch_count % 500 == 0:
            show_learning_progress(epoch, processed_pairs, recent_loss / recent_pairs)
            recent_loss, recent_pairs = 0., 0
    if pair_count == 0:
        raise ValueError('No training word/context pairs survived subsampling.')
    if recent_pairs:
        show_learning_progress(epoch, processed_pairs, recent_loss / recent_pairs)
    training_history.append({'epoch': epoch + 1, 'positive pairs': pair_count,
                             'mean training loss': total_loss / pair_count})
    print(f'Epoch {epoch + 1}/{EPOCHS}: {pair_count:,} pairs; loss {total_loss / pair_count:.4f}', flush=True)
word2vec.eval()
learned_vectors = word2vec.input_embeddings.weight.detach().cpu().numpy().copy()
with torch.no_grad():
    final_pair_loss = word2vec(
        reference_pairs[:, 0], reference_pairs[:, 1], reference_negatives).item()
display(pd.DataFrame(training_history))
display(pd.Series({'before training': initial_pair_loss, 'after training': final_pair_loss},
                  name='Loss on the same fixed training pairs'))

## 6. Choose the words you want to explore

The model has learned vectors for many words. **Edit `chosen_words` below** to choose your own examples. Enter lowercase words, as we did when preparing the training text. A short list keeps the picture readable.

Try words you expect to be related, together with a few you expect to be different. The supplied list is only a starting point, not a required set.

We take each vector directly from the model we just trained. We can only plot words the model learned. The code lists any missing words and leaves them out; at least three known words are needed for this activity.

The trained model stays in memory. It is not saved automatically, so restarting the kernel means training it again.

In [ ]:
chosen_words = [
    'prize', 'reward', 'award', 'gift', 'bonus',
    'meeting', 'appointment', 'team', 'staff', 'office',
    'payment', 'invoice', 'bill', 'receipt', 'refund',
]
if len(set(chosen_words)) != len(chosen_words):
    raise ValueError('Choose each word only once.')
unknown_words = [word for word in chosen_words if word not in word_to_index]
if unknown_words:
    print('Not learned; left out of the map:', ', '.join(unknown_words))
map_words = [word for word in chosen_words if word in word_to_index]
if len(map_words) < 3:
    raise ValueError('Choose at least three words that the model learned.')
selected_vectors = learned_vectors[[word_to_index[word] for word in map_words]].astype(np.float64)
print(f'{len(map_words)} chosen words; {selected_vectors.shape[1]} learned numbers per word')
display(pd.Series(map_words, name='Words included in the map'))

## 7. Check similarity using all 100 numbers

**Cosine similarity** compares the directions of two word vectors. A higher score means they point in more similar directions. This can suggest related word use, but it does not prove that the words mean the same thing.

We calculate these scores using **all 100 numbers per word**, before reducing them to a flat picture. Later, we can check whether the picture gives a similar impression.

First, give every vector length 1. This is called **normalization**. Here, length means the vector's numerical size, not the number of entries: each word still has 100 numbers. This prevents larger vector lengths from dominating the picture.

**Coding task:** divide each row of `selected_vectors` by its length, using `word_lengths[:, None]` to give NumPy one length per row.

The table compares every chosen word with every other chosen word. Each row and column has a word label. The diagonal is 1 because a word is compared with itself; look at the other entries to compare different words.

In [ ]:
word_lengths = np.linalg.norm(selected_vectors, axis=1)
if not np.isfinite(selected_vectors).all() or np.any(word_lengths == 0):
    raise ValueError('Cannot compare vectors with invalid numbers or zero length. Check the training results.')
unit_word_vectors = None  # TODO: divide each row by its own length
word_cosines = cosine_similarity(unit_word_vectors)
display(pd.DataFrame(word_cosines, index=map_words, columns=map_words).round(3))

## 8. Turn 100 numbers into a position on a page

We cannot draw 100 dimensions on a flat page. **Principal Component Analysis (PCA)** finds two directions that show as much of the differences between these word vectors as possible.

The result is two numbers per word: its horizontal and vertical position. Some information is lost.

The title reports how much of the original variation is kept. Here, **variation** means the spread of these selected word vectors. It is **not the percentage of meaning the model understands**.

We use all your chosen, known words to make this picture. This is different from checking a classifier: we are describing a set of words, not testing predictions on new emails.

**Coding task:** use `word_map_pca.fit_transform(unit_word_vectors)` to calculate the two plotting coordinates per word.

In [ ]:
if np.allclose(unit_word_vectors, unit_word_vectors[0]):
    raise ValueError('These word vectors have no visible differences. Choose other words or check training.')
word_map_pca = PCA(n_components=2, svd_solver='full')
word_positions = None  # TODO: calculate two coordinates per normalized word vector
retained_variation = word_map_pca.explained_variance_ratio_.sum()
print(f'Variation kept in the picture: {retained_variation:.1%}')

## 9. Draw your word map

Run the supplied plotting code. Each point comes from a word vector learned by our model and reduced to two numbers by PCA.

The axes do not have names such as 'spam' or 'business'. Nearby points are a clue to explore, not proof that the words have the same meaning.

To try another selection, edit `chosen_words` in section 6 and rerun sections 6–9. **You do not need to train the model again just to change the words in the picture.**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.spines[['top', 'right']].set_visible(False)
ax.scatter(word_positions[:, 0], word_positions[:, 1],
           s=65, color='#00776e', zorder=3)
for word, (x, y) in zip(map_words, word_positions):
    ax.annotate(word, (x, y), xytext=(6, 6), textcoords='offset points', fontsize=11)
ax.set_xlabel('PCA direction 1 (not a named meaning)')
ax.set_ylabel('PCA direction 2 (not a named meaning)')
ax.set_title(f'A map of our chosen words\nTwo directions keep {retained_variation:.1%} of their variation', pad=16)
ax.grid(alpha=0.15)
ax.margins(0.2)
ax.set_aspect('equal', adjustable='box')
fig.text(0.5, 0.02, 'The map loses information. Check cosine scores using all 100 numbers.',
         ha='center', fontsize=9, color='#677786')
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 10. What do the results tell us?

1. Compare the loss before and after training. What did the model learn from? Why does a lower loss not prove better spam detection?
2. Choose two word pairs from your similarity table. Which pair has the higher cosine score? Does the map give the same impression?
3. How much variation does your picture keep? Why does that percentage not tell us how much language the model understands?
4. Would a convincing word map be enough to choose a spam detector for a business? What comparison would you need?

**Your answers:** Replace this sentence with your answers.

## What to remember

- Word2vec learns from nearby words, not spam labels.
- The graph uses the model we trained and the words we chose. There is no separate word-vector file.
- A word has one fixed vector in this model. It cannot give *bank* a different vector in a payment email and a river description.
- Related words are not always interchangeable. Even words with opposite meanings can appear in similar sentences.
- The flat picture loses information. Check cosine scores using all 100 numbers before claiming that two words are close.

A new selection of words can change the PCA picture. Different software versions may also flip the axes, like a mirror image, without changing the relationships. Neither the picture nor the loss tells us how well a spam classifier performs.

For a complete classifier comparison, continue with the [job-category exercise](text_embeddings_exercise.ipynb). That exercise learns from its own training data; do not insert these email-trained vectors into it.

**Keeping your work:** the trained model stays in the notebook's working memory. It is not saved automatically. Restarting the kernel loses that model, even if saved tables and graphs are still visible. To train again from the same starting point, run the notebook from the top. Running only the training loop again continues changing the current model.

Use the email data only within the course's agreed permissions. Do not publish the model's full word list.

## Try it yourself

**Another view of the same model:** change `chosen_words` by adding or removing known words. Run sections 6–9 again and report the new percentage of variation kept. Why can the picture change even though the learned word vectors have not changed? Try an unfamiliar word too and read the message about whether the model learned it.

**Optional next step: spam detection.** Average the known word vectors in each email, then train a classifier using only training emails and their labels. Choose a review threshold using validation emails. Finally, compare it with TF-IDF plus logistic regression on test emails, including false-alarm and missed-spam costs.

If an email has no words the model knows, report that it needs a different handling strategy. Do not quietly give it an all-zero vector and treat the resulting prediction as meaningful.

## References

**Lecture slides:** *Slides_MSC_Data_Science_Week_08_EN.pdf*, page 17, 'A small map of learned word relationships'.

**Instructor sources:** the original Lecture 1 *word2vec_spam_detection.ipynb*, sections 7.1 and 9.4–9.6, and *word2vec_training.py*. We reuse their word-pair preparation and training method. The email-reading code is shared with the Neural Networks material. See the [data notes](../../../00_Data/nlp_data_readme.md) for source and permission details.

**Official documentation, if you want to explore the code:** [PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [SparseAdam](https://docs.pytorch.org/docs/stable/generated/torch.optim.SparseAdam.html), [scikit-learn PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) and [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html).

**Optional reading:** Mikolov et al. (2013), *Distributed Representations of Words and Phrases and their Compositionality*. https://arxiv.org/abs/1310.4546

### Submission details

Include your name, submission date and a short description of your changes.